
# KNN (K-Nearest Neighbors)
### Material introductorio para alumnos de Ingeniería Mecatrónica

---

## Objetivo
Comprender de manera sencilla qué es el algoritmo **K-Nearest Neighbors (KNN)**, cómo funciona y cómo puede utilizarse en problemas básicos de clasificación dentro de Ingeniería Mecatrónica.

## ¿Qué aprenderás?
Al finalizar este notebook podrás:

- entender la idea de “vecinos más cercanos”,
- interpretar cómo KNN toma decisiones,
- entrenar un modelo básico con Python,
- hacer predicciones para nuevos casos,
- evaluar el desempeño del modelo,
- identificar ventajas y limitaciones de KNN.

---

## Contexto en Mecatrónica
En Ingeniería Mecatrónica es común tomar decisiones a partir de datos medidos por sensores. Por ejemplo:

- clasificar una condición de operación como **normal** o **con falla**,
- decidir si una pieza está **aceptada** o **defectuosa**,
- identificar si un motor está en estado **estable** o **inestable**,
- clasificar señales o lecturas según su comportamiento.

KNN es un algoritmo muy intuitivo porque clasifica un nuevo dato observando **qué casos parecidos existen cerca de él**.



## 1. ¿Qué es KNN?

KNN significa **K-Nearest Neighbors**, o en español, **K vecinos más cercanos**.

Es un algoritmo de Machine Learning supervisado que se usa principalmente para:

- **clasificación**, y
- **regresión**.

En este notebook nos enfocaremos en **clasificación**.

### Idea principal
Cuando llega un nuevo dato, KNN:

1. mide qué tan cerca está de los datos conocidos,
2. selecciona los **K vecinos más cercanos**,
3. observa a qué clase pertenecen esos vecinos,
4. asigna al nuevo dato la clase más frecuente.

### Ejemplo intuitivo
Si un nuevo caso está rodeado principalmente por ejemplos de la clase “falla”, probablemente también se clasifique como “falla”.



## 2. ¿Qué significa la K?

La letra **K** indica cuántos vecinos se tomarán en cuenta.

Por ejemplo:

- si **K = 1**, solo se observa el vecino más cercano,
- si **K = 3**, se observan los 3 más cercanos,
- si **K = 5**, se observan los 5 más cercanos.

### Importancia de elegir K
- Un valor pequeño puede hacer al modelo muy sensible al ruido.
- Un valor grande puede hacerlo más estable, pero también más general.

Por eso, elegir K es una parte importante del proceso.



## 3. ¿Cómo mide cercanía?

KNN necesita una forma de medir la distancia entre datos.

La distancia más común es la **distancia euclidiana**, que es la distancia recta entre dos puntos.

Si trabajamos con dos variables, como:

- temperatura,
- vibración,

entonces cada observación puede verse como un punto en el plano.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

pd.set_option("display.precision", 3)



## 4. Ejemplo sencillo: temperatura y vibración en un motor

Supongamos que registramos dos variables en un motor:

- **Temperatura (°C)**
- **Vibración (mm/s)**

Y queremos clasificar cada caso como:

- **0 = normal**
- **1 = posible falla**

Los datos son ficticios, pero razonables para fines didácticos.


In [ ]:

temperatura = np.array([28, 30, 31, 32, 34, 35, 36, 38, 39, 40, 42, 44, 46, 48], dtype=float)
vibracion  = np.array([0.9, 1.0, 1.1, 1.2, 1.4, 1.5, 1.6, 1.9, 2.0, 2.2, 2.5, 2.8, 3.1, 3.4], dtype=float)
falla      = np.array([0,   0,   0,   0,   0,   0,   0,   0,   1,   1,   1,   1,   1,   1], dtype=int)

df = pd.DataFrame({
    "Temperatura_C": temperatura,
    "Vibracion_mm_s": vibracion,
    "Falla": falla
})

df



## 5. Visualización de los datos


In [ ]:

plt.figure(figsize=(8,5))
plt.scatter(df["Temperatura_C"], df["Vibracion_mm_s"], c=df["Falla"])
plt.title("Temperatura y vibración del motor")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Vibración (mm/s)")
plt.grid(True)
plt.show()



Cada punto representa una observación.

KNN clasificará un nuevo punto según los datos que se encuentren más cerca de él.



## 6. Preparación de variables


In [ ]:

X = df[["Temperatura_C", "Vibracion_mm_s"]]
y = df["Falla"]

X.head()



## 7. Entrenamiento del modelo KNN

Entrenaremos primero un modelo con:

- **K = 3**

Esto significa que el modelo tomará en cuenta los **3 vecinos más cercanos**.


In [ ]:

modelo_knn = KNeighborsClassifier(n_neighbors=3)
modelo_knn.fit(X, y)

print("Modelo KNN entrenado correctamente.")



## 8. Predicciones sobre los mismos datos


In [ ]:

y_pred = modelo_knn.predict(X)

resultado = df.copy()
resultado["Prediccion"] = y_pred
resultado



## 9. Evaluación básica


In [ ]:

acc = accuracy_score(y, y_pred)
matriz = confusion_matrix(y, y_pred)

print(f"Accuracy: {acc:.4f}")
print("\nMatriz de confusión:")
print(matriz)


In [ ]:

print(classification_report(y, y_pred, zero_division=0))



### Interpretación sencilla

- **Accuracy** indica la proporción de aciertos.
- La **matriz de confusión** muestra cuántos casos se clasificaron correctamente o incorrectamente.
- El **classification report** resume métricas útiles para analizar el modelo.



## 10. Predicción de un nuevo caso

Supongamos un motor con:

- temperatura = **41 °C**
- vibración = **2.4 mm/s**

Queremos saber si se clasifica como normal o como posible falla.


In [ ]:

nuevo_caso = pd.DataFrame({
    "Temperatura_C": [41],
    "Vibracion_mm_s": [2.4]
})

pred_nuevo = modelo_knn.predict(nuevo_caso)[0]
prob_nuevo = modelo_knn.predict_proba(nuevo_caso)[0]

print("Predicción:", pred_nuevo)
print("Probabilidades [Normal, Falla]:", prob_nuevo)



Si el resultado es:

- **0** → el modelo considera que el motor está normal,
- **1** → el modelo considera posible falla.

Las probabilidades muestran la proporción de vecinos cercanos que pertenecen a cada clase.



## 11. Importancia del escalado de variables

KNN depende directamente de las distancias, por lo que la escala de las variables importa mucho.

Por ejemplo:

- temperatura puede estar entre 20 y 100,
- vibración puede estar entre 0 y 5.

Si no escalamos, una variable con números más grandes puede dominar la distancia.

Por eso frecuentemente se usa **StandardScaler**.


In [ ]:

scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)

modelo_knn_escalado = KNeighborsClassifier(n_neighbors=3)
modelo_knn_escalado.fit(X_escalado, y)

y_pred_escalado = modelo_knn_escalado.predict(X_escalado)

print("Accuracy con escalado:", accuracy_score(y, y_pred_escalado))



## 12. Predicción escalando un nuevo caso


In [ ]:

nuevo_caso_escalado = scaler.transform(nuevo_caso)

pred_nuevo_escalado = modelo_knn_escalado.predict(nuevo_caso_escalado)[0]
prob_nuevo_escalado = modelo_knn_escalado.predict_proba(nuevo_caso_escalado)[0]

print("Predicción con escalado:", pred_nuevo_escalado)
print("Probabilidades [Normal, Falla]:", prob_nuevo_escalado)



## 13. Comparación con distintos valores de K

Probemos ahora distintos valores de K para ver cómo cambian las predicciones.


In [ ]:

valores_k = [1, 3, 5, 7]
resultados_k = []

for k in valores_k:
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit(X_escalado, y)
    pred = modelo.predict(X_escalado)
    acc_k = accuracy_score(y, pred)
    resultados_k.append((k, acc_k))

pd.DataFrame(resultados_k, columns=["K", "Accuracy"])



Esto permite observar que el valor de **K** afecta el comportamiento del modelo.

En la práctica, no siempre existe un valor universalmente mejor; depende de los datos.



## 14. Visualización conceptual de vecinos cercanos

Podemos dibujar el nuevo punto junto con los datos existentes.


In [ ]:

plt.figure(figsize=(8,5))
plt.scatter(df["Temperatura_C"], df["Vibracion_mm_s"], c=df["Falla"], label="Datos conocidos")
plt.scatter(nuevo_caso["Temperatura_C"], nuevo_caso["Vibracion_mm_s"], marker="X", s=200, label="Nuevo caso")
plt.title("Nuevo caso y datos cercanos")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Vibración (mm/s)")
plt.grid(True)
plt.legend()
plt.show()



La idea de KNN es justamente observar qué puntos conocidos están más cerca del nuevo caso y usar sus clases como referencia.



## 15. Otro ejemplo breve: inspección de piezas

Supongamos ahora una clasificación de piezas según:

- **ancho**
- **peso**

Clases:

- **0 = aceptada**
- **1 = defectuosa**


In [ ]:

ancho = np.array([9.8, 10.0, 10.1, 10.2, 10.3, 10.5, 10.7, 10.9, 11.0, 11.2], dtype=float)
peso  = np.array([48, 49, 49.5, 50, 50.5, 52, 53.5, 55, 56, 58], dtype=float)
defecto = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1], dtype=int)

df2 = pd.DataFrame({
    "Ancho_mm": ancho,
    "Peso_g": peso,
    "Defecto": defecto
})

X2 = df2[["Ancho_mm", "Peso_g"]]
y2 = df2["Defecto"]

scaler2 = StandardScaler()
X2_escalado = scaler2.fit_transform(X2)

modelo2 = KNeighborsClassifier(n_neighbors=3)
modelo2.fit(X2_escalado, y2)

pred2 = modelo2.predict(X2_escalado)

print("Accuracy ejemplo 2:", accuracy_score(y2, pred2))
df2["Prediccion"] = pred2
df2



## 16. Ventajas de KNN

- Es fácil de entender.
- No necesita una fase compleja de entrenamiento.
- Puede funcionar bien en problemas simples.
- Es útil para introducir la idea de similitud entre datos.

## 17. Limitaciones de KNN

- Puede ser lento con muchos datos.
- Es sensible a la escala de las variables.
- Puede verse afectado por ruido.
- Elegir K no siempre es trivial.
- Puede tener dificultades si hay muchas variables o clases muy mezcladas.



## 18. Conclusión

KNN es un algoritmo sencillo e intuitivo que clasifica nuevos casos con base en los ejemplos más parecidos.

En Ingeniería Mecatrónica puede ser útil para:

- clasificación de condiciones de operación,
- inspección de calidad,
- detección básica de fallas,
- identificación de estados del sistema.

Además, ayuda a desarrollar intuición sobre cómo los modelos de Machine Learning utilizan la cercanía entre datos para tomar decisiones.



# Ejercicios propuestos

## Ejercicio 1. Comprensión conceptual
Responde con tus propias palabras:

1. ¿Qué significa K en KNN?
2. ¿Cómo decide KNN la clase de un nuevo dato?
3. ¿Por qué es importante escalar las variables en este algoritmo?

---

## Ejercicio 2. Sistema térmico
Crea un conjunto de datos con:

- temperatura,
- corriente,
- clase `0 = normal`, `1 = alarma`.

Después:

1. crea un DataFrame,
2. escala las variables,
3. entrena un modelo KNN,
4. realiza una predicción para un nuevo caso,
5. interpreta el resultado.

---

## Ejercicio 3. Inspección de piezas
Simula datos de piezas usando dos variables:

- largo,
- peso,

y clasifica como:

- `0 = aceptada`
- `1 = defectuosa`.

Realiza:

1. entrenamiento con KNN,
2. evaluación básica,
3. predicción de nuevas piezas,
4. explicación técnica del resultado.

---

## Ejercicio 4. Comparación de K
Usa el mismo conjunto de datos y entrena modelos con:

- `K = 1`
- `K = 3`
- `K = 5`
- `K = 7`

Compara:

1. accuracy,
2. estabilidad del modelo,
3. cuál valor parece más adecuado.

---

## Ejercicio 5. Reflexión aplicada
Menciona un caso real de Ingeniería Mecatrónica donde usarías KNN y explica:

- qué variables medirías,
- qué clases tendrías,
- por qué el algoritmo sería útil.



# Actividad integradora sugerida

## Mini proyecto
Plantea un problema simple de clasificación relacionado con mecatrónica, por ejemplo:

- detección de falla en un motor,
- clasificación de piezas,
- identificación de estados de operación,
- condición segura o peligrosa en un sistema.

Entrega:

1. tabla de datos,
2. descripción del problema,
3. escalado de variables,
4. entrenamiento del modelo KNN,
5. evaluación básica,
6. predicción de al menos dos casos nuevos,
7. conclusión técnica breve.
